# Эксперимент 18: OpenL3 (или fallback) эмбеддинги + классификатор

**Статья:** OpenL3: Deep audio and image embeddings (OpenL3: глубокие аудио- и визуальные эмбеддинги) 2019

**Ссылка:** [https://openl3.readthedocs.io/](https://openl3.readthedocs.io/)

**Краткое описание модели:** OpenL3-эмбеддинги (или fallback mel-статистики) -> классический ML-классификатор.

**Содержание статьи:** OpenL3 предлагает переносимые self-supervised эмбеддинги, которые часто работают без глубокой донастройки. Для малых датасетов это полезно как сильный feature-based baseline. Эксперимент оценивает качество такого представления на задаче дефектов речи.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import time
import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))
from shared import config, data_utils
from shared.results_utils import save_result_csv

import openl3
import soundfile as sf

I0000 00:00:1774633083.522569   50515 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Разбиения и извлечение эмбеддингов

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
print(f"OpenL3 доступен: {True}")

Train: 1942, Val: 417, Test: 417
OpenL3 доступен: True


In [3]:
def extract_openl3_aggregated(path):
    """OpenL3: аудио → (T, D) → mean и max по времени → 2*D."""
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    emb, _ = openl3.get_audio_embedding(audio, sr, embedding_size=512, hop_size=0.1)
    if emb is None or emb.size == 0:
        return np.zeros(512 * 2, dtype=np.float32)
    mean_ = emb.mean(axis=0).astype(np.float32)
    max_  = emb.max(axis=0).astype(np.float32)
    return np.concatenate([mean_, max_])

def extract_fallback_embedding(path):
    """Fallback: mel (n_mels, T) → mean и std по времени → n_mels*2."""
    mel = data_utils.extract_mel_spectrogram(path, max_frames=None)
    mean_ = mel.mean(axis=1).astype(np.float32)
    std_  = mel.std(axis=1).astype(np.float32)
    std_  = np.where(std_ < 1e-8, 1.0, std_)
    return np.concatenate([mean_, std_])

def build_embedding_matrix(paths, letters, use_openl3):
    if use_openl3:
        rows = [extract_openl3_aggregated(p) for p in tqdm.tqdm(paths, desc="OpenL3")]
    else:
        rows = [extract_fallback_embedding(p) for p in tqdm.tqdm(paths, desc="Fallback mel")]
    X = np.stack(rows)
    if letters is not None and letters.shape[1] > 0:
        X = np.hstack([X, letters.astype(np.float32)])
    return X

use_openl3 = True
print("Train...")
X_train = build_embedding_matrix(paths_train, letters_train, use_openl3)
print("Val...")
X_val   = build_embedding_matrix(paths_val, letters_val, use_openl3)
print("Test...")
X_test  = build_embedding_matrix(paths_test, letters_test, use_openl3)
print(f"Признаков на объект: {X_train.shape[1]}")

Train...


OpenL3:   0%|          | 0/1942 [00:00<?, ?it/s]W0000 00:00:1774633095.724019   50515 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


5/5 ━━━━━━━━━━━━━━━━━━━━ 26s 5s/step


OpenL3:   0%|          | 1/1942 [00:30<16:14:31, 30.12s/it]

6/6 ━━━━━━━━━━━━━━━━━━━━ 29s 5s/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step


OpenL3: 100%|█████████▉| 416/417 [2:02:11<00:11, 11.92s/it]

5/5 ━━━━━━━━━━━━━━━━━━━━ 26s 5s/step


OpenL3: 100%|██████████| 417/417 [2:02:39<00:00, 17.65s/it]

Признаков на объект: 1032


## 2. Обучение SVM и оценка

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight=class_weight, probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C": [0.1, 1.0, 10.0], 
    "clf__gamma": ["scale", "auto"]
}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
t0 = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - t0
clf = grid.best_estimator_
print("Лучшие параметры:", grid.best_params_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Лучшие параметры: {'clf__C': 1.0, 'clf__gamma': 'scale'}


In [ ]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=1)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=1)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=1)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
print(f"Accuracy: {accuracy:.4f}, F1 macro: {f1_macro:.4f}, ROC-AUC: {roc_auc:.4f}")

embed_name = "OpenL3" if use_openl3 else "fallback_mel"
save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_18_openl3",
    experiment_name=f"{embed_name} + SVM",
    model="SVM (RBF)",
    accuracy=accuracy,
    f1_macro=f1_macro,
    f1_bad=f1_bad,
    roc_auc=roc_auc,
    precision_bad=precision_bad,
    recall_bad=recall_bad,
    notes=f"embedding={embed_name}, agg=mean+max" if use_openl3 else "embedding=mel mean+std (no openl3)",
    num_params=None,
    train_time_sec=train_time_sec,
)

              precision    recall  f1-score   support

        good       0.84      0.83      0.84       282
         bad       0.65      0.67      0.66       135

    accuracy                           0.78       417
   macro avg       0.75      0.75      0.75       417
weighted avg       0.78      0.78      0.78       417

Accuracy: 0.7794, F1 macro: 0.7500, ROC-AUC: 0.8171
Результат сохранён в result.csv текущего эксперимента
